In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)



In [ ]:
# 1. Carga y Preparación (Foco Transversal)
# Cargamos el set que analiza a los sujetos en un único momento del tiempo [9]
df = pd.read_csv('oasis_cross-sectional.csv')

# Selección de variables predictoras clínicas y demográficas [10]
features = ['Sex', 'Age', 'Educ', 'SES', 'MMSE', 'eTIV', 'nWBV', 'ASF']
X = df[features].copy()

# Codificación de variable binaria (Sexo) [11]
X['Sex'] = X['Sex'].map({'M': 0, 'F': 1})

# --- LINEAMIENTO: Binarización del Target (CDR) ---
# CDR 0 -> 0 ("non demented"), CDR >= 0.5 -> 1 ("demented") [Instrucción]
y = df['CDR'].apply(lambda x: 1 if x >= 0.5 else 0)


In [ ]:
# El dataset transversal tiene nulos en SES y Educ [12, 13]. 
# Imputamos por mediana para no perder muestras valiosas [14].
imputer = SimpleImputer(strategy='median')
X_prep = imputer.fit_transform(X)

# 3. Configuración del Diseño Experimental (10 Validaciones 80/20)
# Creamos contenedores para las métricas de las 10 repeticiones [7]
results = {'acc': [], 'prec': [], 'rec': [], 'f1': [], 'auc': []}
fig, axes = plt.subplots(2, 5, figsize=(22, 10))
axes = axes.flatten()

print("Iniciando 10 validaciones independientes para Naive Bayes...")

for i in range(10):
    # Partición aleatoria única por iteración (80% entrenamiento / 20% prueba) [7]
    X_train, X_test, y_train, y_test = train_test_split(
        X_prep, y, test_size=0.2, random_state=i
    )
    
    # 4. Entrenamiento del Modelo Gaussian Naive Bayes
    # Este modelo es ideal para variables biomédicas continuas [4, 5]
    nb = GaussianNB()
    nb.fit(X_train, y_train)
    
    # Predicciones
    y_pred = nb.predict(X_test)
    y_proba = nb.predict_proba(X_test)[:, 1] # Probabilidades para AUC [15]
    
    # 5. Cálculo y Registro de Métricas [8]
    results['acc'].append(accuracy_score(y_test, y_pred))
    results['prec'].append(precision_score(y_test, y_pred))
    results['rec'].append(recall_score(y_test, y_pred))
    results['f1'].append(f1_score(y_test, y_pred))
    results['auc'].append(roc_auc_score(y_test, y_proba))
    
    # 6. Visualización: Matriz de Confusión por iteración [16]
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Dem.', 'Dem.'])
    disp.plot(ax=axes[i], cmap='Reds', colorbar=False)
    axes[i].set_title(f"Iteración {i+1}\nF1: {results['f1'][-1]:.3f}")

plt.suptitle("Matrices de Confusión (2 Clases) - Naive Bayes Transversal", fontsize=18)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# 7. Reporte Estadístico Final (Media ± DE) [8, 17]
print("\n" + "="*50)
print(f"--- DESEMPEÑO FINAL NAIVE BAYES (10 ITERACIONES) ---")
print(f"{'Métrica':<20} | {'Promedio ± DE':<20}")
print("-" * 50)
for m in results:
    print(f"{m.upper():<20} | {np.mean(results[m]):.4f} ± {np.std(results[m]):.4f}")
print("="*50)